In [ ]:
# Configure the repository root once so the same notebook works locally and on Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import wilcoxon

SCORES_RESULT = str(DATA_DIR / "scores")

# Định nghĩa danh sách mô hình chuẩn
MODELS = ["knn", "dropoutnet", "aldi", "usim", "semproj"]
TRAINED = ["dropoutnet", "aldi", "usim", "semproj"]

def load(model, inp, group=None):
    """Nạp tệp điểm NDCG@20 của từng học viên theo quy ước tên tệp."""
    filename = f"user_ndcg20_{model}_{inp}" + (f"_{group}" if group else "") + ".npy"
    path = os.path.join(SCORES_RESULT, filename)

    if os.path.exists(path):
        arr = np.load(path)
        # Kiểm tra assert kích thước dữ liệu hợp lệ (3340 cho toàn tập, hoặc 1028/2844 cho phân nhóm)
        if group is None:
            assert arr.shape == (3340,), f"Lỗi kích thước toàn tập của {filename}: {arr.shape} (Kỳ vọng: (3340,))"
        elif group == "trung":
            assert arr.shape == (1028,), f"Lỗi kích thước nhóm trùng của {filename}: {arr.shape} (Kỳ vọng: (1028,))"
        elif group == "moi":
            assert arr.shape == (2844,), f"Lỗi kích thước nhóm mới của {filename}: {arr.shape} (Kỳ vọng: (2844,))"
        return arr
    return None


def compare(a, b, n_boot=10000, seed=0):
    """Thực hiện kiểm định Wilcoxon, tính chênh lệch trung bình và Bootstrap CI 95%."""
    assert a is not None and b is not None, "Dữ liệu so sánh bị thiếu (None)!"
    assert a.shape == b.shape, f"Kích thước mảng không khớp: {a.shape} vs {b.shape}"
    
    d = a - b
    # Kiểm định Wilcoxon hai phía 
    p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided").pvalue
    
    rng = np.random.default_rng(seed)
    boots = np.array(
        [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
    )
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi), int((d != 0).sum()), float(p)


def holm(pvals):
    """Hiệu chỉnh p-value theo phương pháp Holm."""
    pvals = np.asarray(pvals, dtype=np.float64)
    m = len(pvals)
    adj = np.empty(m, dtype=np.float64)
    run = 0.0
    for rank, i in enumerate(np.argsort(pvals)):
        run = max(run, (m - rank) * pvals[i])
        adj[i] = min(1.0, run)
    return adj

def family(pairs):
    """Chạy họ kiểm định cho một tập hợp các cặp so sánh mô hình."""
    rows = {}
    p_vals = []
    for k, (a, b) in pairs.items():
        res = compare(a, b)
        if res is not None:
            rows[k] = res
            p_vals.append(res[4])
        else:
            rows[k] = (0, 0, 0, 0, 1.0)
            p_vals.append(1.0)
            
    p_adj = holm(p_vals) if p_vals else []
    data = []
    for (k, r), pa in zip(rows.items(), p_adj):
        data.append((k, r[0], r[1], r[2], r[3], r[4], pa))
    return pd.DataFrame(data, columns=["model", "diff", "lo", "hi", "n_khac_0", "p", "p_holm"])

def pairs_of(inp_a, inp_b, group=None):
    """Lấy các cặp điểm so sánh giữa hai loại đầu vào cho tất cả các mô hình có sẵn."""
    res_pairs = {}
    for m in MODELS:
        a_arr = load(m, inp_a, group)
        b_arr = load(m, inp_b, group)
        if a_arr is not None and b_arr is not None:
            res_pairs[m] = (a_arr, b_arr)
    return res_pairs

print("Đã thiết lập thành công các hàm load, compare, wilcoxon, holm và assert kiểm chứng dữ liệu!")

Đã thiết lập thành công các hàm load, compare, wilcoxon, holm và assert kiểm chứng dữ liệu!


In [9]:
# 1. ĐÁNH GIÁ E-RQ1: Thêm hồ sơ sư phạm LLM có cải thiện cold-start không?
print("================ CHẠY ĐÁNH GIÁ E-RQ1 ================")
e1_main = family(pairs_of("fused", "raw"))   # Gốc + LLM so với Văn bản gốc (Chính)
e1_llm  = family(pairs_of("llm", "raw"))     # Chỉ hồ sơ LLM so với Văn bản gốc (Phụ)

def verdict_erq1(df):
    pos_sig = (df["diff"] > 0) & (df["p_holm"] < 0.05)
    tr = df[df["model"].isin(TRAINED)]
    knn_worse = ((df["model"] == "knn") & (df["diff"] < 0) & (df["p_holm"] < 0.05)).any()
    if len(tr) and pos_sig[tr.index].all() and not knn_worse:
        return "Cải thiện nhất quán"
    return "Phụ thuộc kiến trúc" if pos_sig.any() else "Không có bằng chứng"

print("\n--- E-RQ1 Chính (Gốc + LLM vs Văn bản gốc) ---")
print(e1_main)
print(f"=> Kết luận E-RQ1 Chính: {verdict_erq1(e1_main)}")

print("\n--- E-RQ1 Phụ (Chỉ LLM vs Văn bản gốc) ---")
print(e1_llm)
print(f"=> Kết luận E-RQ1 Phụ: {verdict_erq1(e1_llm)}")

================ CHẠY ĐÁNH GIÁ E-RQ1 ================

--- E-RQ1 Chính (Gốc + LLM vs Văn bản gốc) ---
        model      diff        lo        hi  n_khac_0             p  \
0         knn  0.000060 -0.003268  0.003378       639  9.867543e-01   
1  dropoutnet  0.010253  0.006413  0.014061      1504  1.252261e-19   

         p_holm  
0  9.867543e-01  
1  2.504522e-19  
=> Kết luận E-RQ1 Chính: Cải thiện nhất quán

--- E-RQ1 Phụ (Chỉ LLM vs Văn bản gốc) ---
        model      diff        lo        hi  n_khac_0             p  \
0         knn -0.023413 -0.028898 -0.017987       821  1.807341e-18   
1  dropoutnet -0.009011 -0.013696 -0.004370      1576  2.193439e-01   

         p_holm  
0  3.614682e-18  
1  2.193439e-01  
=> Kết luận E-RQ1 Phụ: Không có bằng chứng


In [10]:
# 2. ĐÁNH GIÁ E-RQ2: Lợi ích nằm ở khóa mới hoàn toàn hay khóa trùng?
print("\n================ CHẠY ĐÁNH GIÁ E-RQ2 ================")
e2 = {g: family(pairs_of("fused", "raw", g)) for g in ["moi", "trung"]}

print("\n--- E-RQ2 trên Nhóm Khóa Mới Hoàn Toàn ---")
print(e2["moi"])

print("\n--- E-RQ2 trên Nhóm Khóa Trùng ---")
print(e2["trung"])


================ CHẠY ĐÁNH GIÁ E-RQ2 ================

--- E-RQ2 trên Nhóm Khóa Mới Hoàn Toàn ---
        model      diff        lo        hi  n_khac_0             p  \
0         knn  0.005313  0.002472  0.008176       388  5.796887e-04   
1  dropoutnet  0.014659  0.010721  0.018596      1041  3.660604e-28   

         p_holm  
0  5.796887e-04  
1  7.321207e-28  

--- E-RQ2 trên Nhóm Khóa Trùng ---
        model      diff        lo        hi  n_khac_0         p    p_holm
0         knn -0.019130 -0.027310 -0.010946       264  0.000002  0.000004
1  dropoutnet  0.000077 -0.007653  0.007787       547  0.356620  0.356620


In [12]:
import os
import pandas as pd

# 1. Đảm bảo thư mục lưu trữ kết quả tồn tại
os.makedirs("results_semproj", exist_ok=True)

# 2. Tổng hợp các bảng kết quả kiểm định từ hàm family()
df_erq1_main = family(pairs_of("fused", "raw"))   # E-RQ1 Chính: Gốc + LLM so với Văn bản gốc
df_erq1_llm  = family(pairs_of("llm", "raw"))     # E-RQ1 Phụ: Chỉ hồ sơ LLM so với Văn bản gốc

# Thêm cột nhận diện câu hỏi nghiên cứu để tiện phân loại khi lưu file CSV chung
df_erq1_main.insert(0, "Research_Question", "E-RQ1_Main (Fused vs Raw)")
df_erq1_llm.insert(0, "Research_Question", "E-RQ1_Sub (LLM vs Raw)")

# Nếu bạn đã chạy E-RQ2 (phân nhóm moi / trung), có thể gom tiếp vào đây:
e2_moi = family(pairs_of("fused", "raw", "moi"))
e2_moi.insert(0, "Research_Question", "E-RQ2_Moi (Fused vs Raw)")
e2_trung = family(pairs_of("fused", "raw", "trung"))
e2_trung.insert(0, "Research_Question", "E-RQ2_Trung (Fused vs Raw)")

# Gộp tất cả các bảng kiểm định lại thành một DataFrame duy nhất
df_all_tests = pd.concat([df_erq1_main, df_erq1_llm, e2_moi, e2_trung], ignore_index=True)

# 3. Trình bày và in kết quả ra màn hình một cách rõ ràng, chuyên nghiệp
print("==================================================================================")
print("             BẢNG TỔNG HỢP KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ (WILCOXON + HOLM)")
print("==================================================================================")
print(df_all_tests.to_string(index=False))
print("==================================================================================")

# 4. Lưu toàn bộ kết quả kiểm định vào tệp CSV theo đúng yêu cầu
output_csv_path = "results_semproj/statistical_tests_wilcoxon_holm.csv"
df_all_tests.to_csv(output_csv_path, index=False)
print(f"\n[Thành công] Đã lưu toàn bộ bảng kết quả kiểm định vào: {output_csv_path}")

             BẢNG TỔNG HỢP KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ (WILCOXON + HOLM)
         Research_Question      model      diff        lo        hi  n_khac_0            p       p_holm
 E-RQ1_Main (Fused vs Raw)        knn  0.000060 -0.003268  0.003378       639 9.867543e-01 9.867543e-01
 E-RQ1_Main (Fused vs Raw) dropoutnet  0.010253  0.006413  0.014061      1504 1.252261e-19 2.504522e-19
    E-RQ1_Sub (LLM vs Raw)        knn -0.023413 -0.028898 -0.017987       821 1.807341e-18 3.614682e-18
    E-RQ1_Sub (LLM vs Raw) dropoutnet -0.009011 -0.013696 -0.004370      1576 2.193439e-01 2.193439e-01
  E-RQ2_Moi (Fused vs Raw)        knn  0.005313  0.002472  0.008176       388 5.796887e-04 5.796887e-04
  E-RQ2_Moi (Fused vs Raw) dropoutnet  0.014659  0.010721  0.018596      1041 3.660604e-28 7.321207e-28
E-RQ2_Trung (Fused vs Raw)        knn -0.019130 -0.027310 -0.010946       264 2.021696e-06 4.043393e-06
E-RQ2_Trung (Fused vs Raw) dropoutnet  0.000077 -0.007653  0.007787       547 3.566199e-01 3.566